# Seed and Fold Sensitivity

This notebook reruns the cross-validated evaluation of every target and model family across random seeds 42, 123 and 2026 and outer CV fold counts 5, 7 and 10. It reports pooled out-of-fold R², RMSE and MAE for each combination.

## How to run

Run the cells from top to bottom, or choose **Run All**. The setup cell finds the project root and checks that the processed phenotype and genotype files exist; if either is missing, run the data-preparation workflow first.

To test a different grid, edit `SEEDS` and `FOLD_COUNTS` in the setup cell. Fold counts must be between the pipeline's configured minimum and the number of labeled clones.

The full sweep runs 135 model evaluations (5 targets × 3 models × 3 seeds × 3 fold counts) and may take several minutes. Results are saved to `outputs/model_sensitivity_by_seed_and_folds.csv` and `outputs/model_sensitivity_summary.csv`.

In [1]:
# Load necessary libraries and set up project paths
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "capstone_pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate src/capstone_pipeline.py.")

sys.path.insert(0, str(ROOT / "src"))
import capstone_pipeline as cp

PHENOTYPE_PATH = ROOT / "data" / "processed" / "phenotype_clean.csv"
GENOTYPE_PATH = ROOT / "data" / "processed" / "genotype_clean.csv"
OUTPUT_DIR = ROOT / "outputs"
# Change these tuples to compare a different reproducible test grid.
SEEDS = (42, 123, 2026)
FOLD_COUNTS = (5, 7, 10)

for path in (PHENOTYPE_PATH, GENOTYPE_PATH):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required processed input not found: {path}. "
            "Run notebook 02 or `python src/etl_pipeline.py` first."
        )

print(f"Project root: {ROOT}")
print(f"Seeds: {SEEDS}; outer folds: {FOLD_COUNTS}")

Project root: C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0
Seeds: (42, 123, 2026); outer folds: (5, 7, 10)


In [2]:
# Load and validate phenotype and genotype inputs
phenotype, genotype = cp.load_inputs(PHENOTYPE_PATH, GENOTYPE_PATH)
marker_columns = [column for column in genotype.columns if column != "clone_id"]
labeled = phenotype.merge(genotype, on="clone_id", how="inner").set_index("clone_id")
targets = [
    target for target in cp.TARGET_COLUMNS
    if cp.TARGET_SOURCE_COLUMNS[target] in labeled.columns
]
cp.validate_inputs(
    phenotype, genotype,
    targets=[cp.TARGET_SOURCE_COLUMNS[target] for target in targets],
)
if labeled.empty:
    raise ValueError("No phenotype/genotype clone IDs overlap.")

print(f"Labeled clones: {len(labeled)}")
print(f"Markers: {len(marker_columns)}")
print(f"Targets: {targets}")

Labeled clones: 54
Markers: 500
Targets: ['market_index', 'dri', 'eqi', 'parental_value', 'ayi']


## Cross-validation sensitivity

Each row is one target, model, seed and outer-fold configuration. The summary table then gives the mean, standard deviation and range of R² across seeds for each target, model and fold count.

In [3]:
# Run every target and model family for each seed and fold count
sensitivity_rows = []
for n_folds in FOLD_COUNTS:
    for seed in SEEDS:
        print(f"Running {n_folds} folds, seed {seed}...", flush=True)
        for target in targets:
            source_column = cp.TARGET_SOURCE_COLUMNS[target]
            for family in cp.MODEL_FAMILIES:
                outcome = cp._fit_and_evaluate_target(
                    target, labeled[marker_columns], labeled[source_column],
                    model_family=family, seed=seed, n_cv_folds=n_folds,
                )
                if outcome.get("skipped"):
                    raise RuntimeError(
                        f"{target}/{family} skipped: {outcome['skip_reason']}"
                    )
                sensitivity_rows.append({
                    "target": target,
                    "model_family": family,
                    "seed": seed,
                    "n_cv_folds": outcome["n_cv_folds"],
                    "cv_r2": outcome["cv_r2"],
                    "cv_rmse": outcome["cv_rmse"],
                    "cv_mae": outcome["cv_mae"],
                })

sensitivity_results = pd.DataFrame(sensitivity_rows)
expected_rows = len(targets) * len(cp.MODEL_FAMILIES) * len(SEEDS) * len(FOLD_COUNTS)
if len(sensitivity_results) != expected_rows:
    raise AssertionError(f"Expected {expected_rows} results; got {len(sensitivity_results)}.")

# Summarize across seeds for each target, model and fold count
sensitivity_summary = (
    sensitivity_results
    .groupby(["target", "model_family", "n_cv_folds"], as_index=False)
    .agg(
        seed_runs=("seed", "nunique"),
        cv_r2_mean=("cv_r2", "mean"),
        cv_r2_sd=("cv_r2", "std"),
        cv_r2_min=("cv_r2", "min"),
        cv_r2_max=("cv_r2", "max"),
        cv_rmse_mean=("cv_rmse", "mean"),
        cv_mae_mean=("cv_mae", "mean"),
    )
)
sensitivity_results.to_csv(OUTPUT_DIR / "model_sensitivity_by_seed_and_folds.csv", index=False)
sensitivity_summary.to_csv(OUTPUT_DIR / "model_sensitivity_summary.csv", index=False)
print(f"Saved {len(sensitivity_results)} sensitivity rows to {OUTPUT_DIR}.")
display(sensitivity_summary.round(3))

Running 5 folds, seed 42...


Running 5 folds, seed 123...


Running 5 folds, seed 2026...


Running 7 folds, seed 42...


Running 7 folds, seed 123...


Running 7 folds, seed 2026...


Running 10 folds, seed 42...


Running 10 folds, seed 123...


Running 10 folds, seed 2026...


Saved 135 sensitivity rows to C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\outputs.


,target,model_family,n_cv_folds,seed_runs,cv_r2_mean,cv_r2_sd,cv_r2_min,cv_r2_max,cv_rmse_mean,cv_mae_mean
0,ayi,elastic_net,5,3,0.933,0.003,0.930,0.936,0.036,0.030
1,ayi,elastic_net,7,3,0.935,0.003,0.933,0.938,0.036,0.030
2,ayi,elastic_net,10,3,0.945,0.003,0.941,0.948,0.033,0.027
3,ayi,mlp_baseline,5,3,0.929,0.004,0.926,0.934,0.038,0.031
4,ayi,mlp_baseline,7,3,0.935,0.004,0.930,0.937,0.036,0.030
5,ayi,mlp_baseline,10,3,0.939,0.008,0.930,0.946,0.035,0.029
6,ayi,ridge,5,3,0.818,0.015,0.804,0.834,0.060,0.049
7,ayi,ridge,7,3,0.824,0.022,0.806,0.849,0.059,0.047
8,ayi,ridge,10,3,0.839,0.009,0.829,0.846,0.056,0.046
9,dri,elastic_net,5,3,0.833,0.005,0.828,0.838,0.064,0.048
